# TP Máquina de Turing Universal
---

**¿Por qué se dice que una MTU es capaz de simular cualquier Máquina de Turing?**

> *Respuesta*
>
> Porque la MTU no tiene "grabado" ningún problema en particular. Lo que hace es leer de su propia cinta la descripción de otra máquina M (sus estados y su tabla de transiciones pasados a binario) y usarla como si fuera un programa: se fija en qué estado está M y qué símbolo lee, busca en la tabla qué tiene que hacer, lo hace, y vuelve a empezar.
>
> Como ese ciclo es siempre el mismo sin importar qué haga M, con una sola máquina U alcanza para simular a cualquiera. Es lo mismo que pasa con una compu: el hardware es uno solo y lo que cambia es el programa que le cargás.

**Suponer que se tiene una MT M que acepta todas las cadenas que terminan en 01. Indicar qué debería hacer una MTU con las siguientes entradas. Explicar en cada caso si la MTU acepta o rechaza y por qué**
* U(⟨M⟩,1101)
* U(⟨M⟩,100)
* U(⟨M⟩,01)
* U(⟨M⟩,111)

> *Respuesta*
>
> La MTU no decide nada por su cuenta: simula a M y da la misma respuesta que daría M. Como M acepta las cadenas que terminan en 01, solo hay que mirar los últimos dos símbolos:
>
> * U(⟨M⟩, 1101): acepta, termina en 01.
> * U(⟨M⟩, 100): rechaza, termina en 00.
> * U(⟨M⟩, 01): acepta, es justo 01.
> * U(⟨M⟩, 111): rechaza, termina en 11.
>
> Más abajo, en la parte del programa, corro estas mismas cuatro cadenas con el simulador para ver que dé lo mismo.

**Dada la siguiente MT M:** 

| Q   | 0        | 1        |
| --  | --       | --       |
| q0  | (q1,1,R) | (q1,0,R) |
| q1  | (qf,0,R) | (qf,1,R) |
| qf  | -        | -        |

1. Explicar que hace M 
2. Explicar qué información debería recibir una MTU para poder simular M
3. Codificar la cintar de MTU sabiendo que configuración de la cinta de MT M es 1 q0 0 1 1

> *Respuesta*
>
> **1. Qué hace M**
>
> M lee dos símbolos y para. En q0 invierte el bit que tiene abajo (0 pasa a 1, 1 pasa a 0), se mueve a la derecha y pasa a q1. En q1 deja el bit como está, se mueve a la derecha y pasa a qf. En qf no hay transiciones, así que ahí se detiene.
>
> O sea: niega el primer símbolo, saltea el segundo y queda con el cabezal dos lugares a la derecha de donde arrancó. El resto de la cinta no lo toca.
>
> **2. Qué tiene que recibir la MTU**
>
> Dos cosas en la cinta:
>
> * La tabla de transiciones de M pasada a binario (⟨M⟩). Ahí adentro van los estados, los símbolos y los movimientos.
> * Cómo está M al arrancar: qué hay en su cinta, dónde está el cabezal y en qué estado está.
>
> Y las dos partes tienen que estar separadas con algún símbolo especial (`$`, `#`, `*`), si no U no sabe dónde termina una cosa y empieza la otra.
>
> **3. Codificar la cinta de la MTU**
>
> La configuración es `1 q0 0 1 1`: la cinta de M tiene `1 0 1 1`, está en q0 y el cabezal está sobre el 0 (el símbolo que viene después de q0).
>
> Codificación, usando 2 bits para todo:
>
> * Estados: q0 = 00, q1 = 01, qf = 10
> * Símbolos: 0 = 00, 1 = 01
> * Movimientos: R = 00, L = 01, S = 10
>
> Cada transición es un registro de 10 bits con `estado · leído · estado siguiente · escrito · movimiento`:
>
> ```
> Transición              Campos           Registro
> δ(q0, 0) = (q1, 1, R)   00 00 01 01 00   0000010100
> δ(q0, 1) = (q1, 0, R)   00 01 01 00 00   0001010000
> δ(q1, 0) = (qf, 0, R)   01 00 10 00 00   0100100000
> δ(q1, 1) = (qf, 1, R)   01 01 10 01 00   0101100100
> ```
>
> Armado de la cinta:
>
> * A la izquierda del `$` va la cinta de M codificada (`01 00 01 01`), pero en el lugar del cabezal se pone un `*` en vez del símbolo: `01*0101`. El 0 (`00`) que estaba ahí no se pierde, se pasa a la derecha del `$`.
> * Entre el `$` y el primer `#` va el registro 0: estado actual (00) + símbolo leído (00) = `0000`.
> * Después los cuatro registros separados con `#`.
>
> ```
> 01*0101$0000#0000010100#0001010000#0100100000#0101100100
> ```

1. **Codificación de una máquina simple**  
   * Definir una máquina $M$ que ...
   * Codificar sus estados, símbolos y transiciones en forma numérica

> *Respuesta*
>
> Pensé una máquina que invierta los bits de una cadena binaria: recorre la cadena cambiando cada 0 por 1 y cada 1 por 0, y cuando llega al blanco vuelve al principio y para. Es como la del punto 3 pero para toda la cadena, y a diferencia de la del punto 2 no acepta ni rechaza, deja un resultado en la cinta. Por ejemplo, 1011 pasa a 0100.
>
> **La máquina**
>
> * q0: va para la derecha invirtiendo cada bit.
> * q1: encontró el blanco del final, vuelve para la izquierda sin tocar nada.
> * qf: llegó al blanco del principio, da un paso a la derecha y para. Queda parada sobre el primer símbolo del resultado.
>
> ```
> Q    0            1            □
> q0   (q0, 1, R)   (q0, 0, R)   (q1, □, L)
> q1   (q1, 0, L)   (q1, 1, L)   (qf, □, R)
> qf   -            -            -
> ```
>
> **Codificación**
>
> Uso 2 bits para todo: estados, símbolos y movimientos. Con 2 bits entran hasta 4 de cada cosa, y esta máquina tiene 3 estados, 3 símbolos (0, 1 y el blanco) y usa 2 movimientos, así que alcanza.
>
> * Estados: q0 = 00, q1 = 01, qf = 10
> * Símbolos: 0 = 00, 1 = 01, □ = 10
> * Movimientos: R = 00, L = 01, S = 10
>
> Cada registro queda de 10 bits: 2 + 2 + 2 + 2 + 2.
>
> ```
> Transición              Campos           Registro
> δ(q0, 0) = (q0, 1, R)   00 00 00 01 00   0000000100
> δ(q0, 1) = (q0, 0, R)   00 01 00 00 00   0001000000
> δ(q0, □) = (q1, □, L)   00 10 01 10 01   0010011001
> δ(q1, 0) = (q1, 0, L)   01 00 01 00 01   0100010001
> δ(q1, 1) = (q1, 1, L)   01 01 01 01 01   0101010101
> δ(q1, □) = (qf, □, R)   01 10 10 10 00   0110101000
> ```
>
> La tabla completa, que es lo que le voy a pasar a la MTU, queda:
>
> ```
> 0000000100#0001000000#0010011001#0100010001#0101010101#0110101000
> ```
>
> Y la entrada también hay que codificarla. Por ejemplo 1011 queda `01 00 01 01`, o sea `01000101`. Con la máquina en q0 y el cabezal en el primer símbolo, la cinta de la MTU es:
>
> ```
> *000101$0001#0000000100#0001000000#0010011001#0100010001#0101010101#0110101000
> ```
>
> El primer 1 (`01`) se reemplaza por el `*` y pasa al registro 0 junto con el estado q0 (`00`).

2. **Simulación básica**  
   * Implementar en Python un programa que reciba:
     * La codificación de una máquina $M$
     * Una cadena de entrada $w$
   * El programa debe simular paso a paso la ejecución de $M$ sobre $w$

> *Respuesta*
>
> El programa está en [mtu.py](mtu.py), en esta misma carpeta. Recibe la cinta completa de la MTU, con la tabla de transiciones de M y la entrada ya codificadas, y simula a M paso a paso.
>
> Cómo ejecutarlo, con cintas de ejemplo para probarlo, está explicado en [instrucciones-mtu.txt](instrucciones-mtu.txt).

3. **Pruebas de funcionamiento**  
   * Probar la simulación con diferentes entradas
   * Documentar los resultados

> *Respuesta*
>
> Primero varias cadenas con la máquina que invierte bits. Para cada una escribí a mano la primera parte de la cinta, que es la entrada con el `*` en el primer símbolo y el registro 0 con el estado 00, y le pego la tabla de transiciones, que es la misma para todas. La cadena vacía queda como un solo blanco (`10`) bajo el cabezal. Para leer el resultado uso un diccionario que traduce los códigos a 0 y 1. La columna "esperado" la calculo directo en Python dando vuelta los caracteres, así la comparación no depende de mí:

In [1]:
from mtu import MTU

tabla = "0000000100#0001000000#0010011001#0100010001#0101010101#0110101000"
de_bits = {"00": "0", "01": "1"}

pruebas = [
    ("1011",   "*000101$0001#"),
    ("0000",   "*000000$0000#"),
    ("1111",   "*010101$0001#"),
    ("1",      "*$0001#"),
    ("0",      "*$0000#"),
    ("",       "*$0010#"),
    ("100110", "*0000010100$0001#"),
    ("0101",   "*010001$0000#"),
]

print(f"{'cadena':<9} {'esperado':<10} {'MTU':<10} {'pasos':<6} coincide")
print("-" * 44)
for w, inicio in pruebas:
    u = MTU(inicio + tabla)
    u.ejecutar(mostrar=False)
    resultado = "".join(de_bits[b] for b in u.salida().split())
    esperado = "".join("1" if c == "0" else "0" for c in w)
    print(f"{w or 'ε':<9} {esperado or 'ε':<10} {resultado or 'ε':<10} {u.pasos:<6} {'sí' if resultado == esperado else 'NO'}")

cadena    esperado   MTU        pasos  coincide
--------------------------------------------
1011      0100       0100       10     sí
0000      1111       1111       10     sí
1111      0000       0000       10     sí
1         0          0          4      sí
0         1          1          4      sí
ε         ε          ε          2      sí
100110    011001     011001     14     sí
0101      1010       1010       10     sí


> Como es una MTU, le puedo pasar cualquier máquina. La del punto 3, con la cinta que armé a mano en ese punto:

In [2]:
u = MTU("01*0101$0000#0000010100#0001010000#0100100000#0101100100")
u.ejecutar()

paso  0:  01*0101$0000#0000010100#0001010000#0100100000#0101100100
          estado 00   cinta de M:  01 [00] 01 01
paso  1:  0101*01$0101#0000010100#0001010000#0100100000#0101100100
          estado 01   cinta de M:  01 01 [01] 01
paso  2:  010101*$1001#0000010100#0001010000#0100100000#0101100100
          estado 10   cinta de M:  01 01 01 [01]


> Y la del punto 2 (termina en 01). Como con 2 bits entran como mucho 4 estados, la armé con q0, q1 y q2, que recuerdan cómo viene terminando la cadena (nada útil, un 0, o 01), y un estado qa de aceptación. No hace falta un estado de rechazo: si llega al blanco estando en q0 o q1 no hay regla, así que para ahí, y como no es qa, rechaza.
>
> ```
> Q    0            1            □
> q0   (q1, 0, R)   (q0, 1, R)   -
> q1   (q1, 0, R)   (q2, 1, R)   -
> q2   (q1, 0, R)   (q0, 1, R)   (qa, □, L)
> qa   -            -            -
> ```
>
> Estados: q0 = 00, q1 = 01, q2 = 10, qa = 11. Símbolos y movimientos igual que antes. La MTU no sabe que `11` es el de aceptación: solo me dice en qué estado paró, y yo me fijo si es qa:

In [3]:
tabla_01 = "0000010000#0001000100#0100010000#0101100100#1000010000#1001000100#1010111001"

pruebas_01 = [
    ("1101", "*010001$0001#"),
    ("100",  "*0000$0001#"),
    ("01",   "*01$0000#"),
    ("111",  "*0101$0001#"),
]

for w, inicio in pruebas_01:
    u = MTU(inicio + tabla_01)
    u.ejecutar(mostrar=False)
    resultado = "acepta" if u.estado == "11" else "rechaza"
    print(f"U(<M>, {w:<5}) => paró en el estado {u.estado} ({u.pasos} pasos) => {resultado}")

U(<M>, 1101 ) => paró en el estado 11 (5 pasos) => acepta
U(<M>, 100  ) => paró en el estado 01 (3 pasos) => rechaza
U(<M>, 01   ) => paró en el estado 11 (3 pasos) => acepta
U(<M>, 111  ) => paró en el estado 00 (3 pasos) => rechaza


> **Resultados**
>
> La inversión de bits dio bien en todos los casos, incluidos los raros: la cadena vacía devuelve vacío, y con un solo símbolo también anda. Los pasos son siempre 2·|w| + 2: recorre la cadena de ida, la vuelve a recorrer de vuelta y hace un paso más en cada punta al tocar los blancos.
>
> La máquina del punto 3, con la cinta que armé a mano, termina en el estado `10` (qf) con `01 01 01 01`, o sea 1111, que es lo que había dicho que hacía. Y las cuatro cadenas del punto 2 dan lo mismo que respondí ahí: 1101 y 01 aceptan porque paran en qa (`11`), y 100 y 111 rechazan porque paran en q1 y q0, que no son de aceptación.
>
> Lo que me parece más importante de esta parte es que las tres máquinas corrieron con el mismo programa sin cambiarle nada, lo único que cambió fue la tabla que le pasé.

4. **Informe final**  
   * Explicar la codificación utilizada
   * Mostrar ejemplos de ejecución
   * Reflexionar sobre la relación entre la MTU y las computadoras modernas

> *Respuesta*
>
> **Codificación**
>
> Usé la estructura de la teoría, pero con 2 bits para todo: cada estado, cada símbolo y cada movimiento (R = 00, L = 01, S = 10) ocupa 2 bits. Cada transición es un registro con los cinco campos pegados, sin separadores, así que todos los registros miden 10 bits. La cinta de la MTU queda `cinta de M $ registro 0 # registros`, con el `*` marcando el cabezal.
>
> Que todos los campos tengan el mismo largo es lo que permite cortar un registro sin ambigüedad: la MTU sabe que cada 2 bits empieza un campo nuevo, y no hace falta decirle nada más. La contra es que con 2 bits entran como mucho 4 estados y 4 símbolos, por eso a la máquina del punto 2 le saqué el estado de rechazo. Para máquinas más grandes habría que usar más bits por campo.
>
> **Ejemplos de ejecución**
>
> Están más arriba. Resumiendo:
>
> ```
> Máquina                           Entrada      Resultado
> M invierte bits (ejercicio 1)     1011         deja 0100 en 10 pasos
> M invierte bits (ejercicio 1)     0000         deja 1111 en 10 pasos
> M del punto 3 (invierte un bit)   1 q0 0 1 1   termina en qf con la cinta 1 1 1 1
> M termina en 01 (punto 2)         1101         acepta en 5 pasos
> M termina en 01 (punto 2)         100          rechaza en 3 pasos
> ```
>
> Lo interesante es que el programa es el mismo para las tres máquinas, lo único que cambia es el string que le paso.
>
> **La MTU y las computadoras de hoy**
>
> Lo que más me sirvió para entenderlo fue pensar a la MTU como una compu y a ⟨M⟩ como un programa. La tabla de M viaja por la cinta igual que la entrada w: no hay diferencia entre "programa" y "datos", son todos bits, y la diferencia la pone U según qué parte de la cinta está mirando. Eso es justo la idea de programa almacenado de von Neumann, instrucciones y datos en la misma memoria.
>
> Comparando:
>
> * La MTU es el procesador: hardware fijo que siempre hace el mismo ciclo (buscar la transición, aplicarla, avanzar). Vendría a ser el fetch-decode-execute.
> * ⟨M⟩ es el programa cargado en memoria.
> * w son los datos de entrada.
> * El registro 0 es algo así como el contador de programa: dice en qué parte de la ejecución estoy.
>
> También se parece a un intérprete: mtu.py no sabe nada de negar bits ni de sufijos 01, igual que Python no sabe nada de un programa hasta que lo lee. De hecho lo que hice es un intérprete de MT en Python, que a su vez lo interpreta el procesador.
>
> Y por el otro lado, como U puede simular cualquier M, uno se tienta con hacer una máquina que diga si M va a parar o no con cierta entrada. Turing demostró que eso no se puede (el problema de la parada). Por eso el simulador tiene un límite de pasos: si M no para, U tampoco, y no hay forma general de saberlo de antemano.